# TD3C FFA — positioning, curve and sentiment

ICE monthly **TDL** (MEG–China VLCC) in **USD/t**. OI lives on `FILE_FLAG=Final` only (Early has settle/volume).

Chart grammar follows the desk crude OI pack (total OI, OI vs days-to-expiry, latest OI + Δ), plus freight-specific panels.

See `docs/ice_ffa_map.md`. Lot = 1,000 t; one VLCC cargo ≈ 270 kt ≈ 270 lots.

In [1]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT.parent / "desk_core"))

from freight.ice import (  # noqa: E402
    asof,
    front_n,
    load_daily_prompt,
    load_ffa_monthly,
    oi_delta,
    total_oi,
)

pd.options.display.float_format = lambda x: f"{x:,.2f}"
CACHE = ROOT / "data" / "cache"
# First run pulls Snowflake; later runs use parquet. Flip refresh after new MFT files land.
REFRESH = True
START = date(2022, 1, 1)
print("ROOT", ROOT)

ROOT c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


In [2]:
tdl = load_ffa_monthly("TDL", start=START, cache_dir=CACHE, refresh=REFRESH)
dfn = load_daily_prompt("DFN", start=START, cache_dir=CACHE, refresh=REFRESH)

# Final rows carry OI (including 0). Early-only latest session is price-only.
oi_ok = tdl.loc[tdl["OI_LOTS"].notna()].copy()
session, snap = asof(oi_ok)
prev_dates = sorted(oi_ok["TRADE_DATE"].unique())
prev_session = prev_dates[-2] if len(prev_dates) >= 2 else session
prev_snap = oi_ok.loc[oi_ok["TRADE_DATE"] == prev_session]
front = front_n(snap, 12, session)
price_session, price_snap = asof(tdl)
price_front = front_n(price_snap, 24, price_session)

display(Markdown(
    f"**OI session** {session} (Final) · **price session** {price_session} · "
    f"strips {tdl['STRIP'].nunique()} · rows {len(tdl):,}"
))
front[["STRIP", "CONTRACT_CODE", "SETTLE_USD_T", "OI_LOTS", "VOLUME_LOTS", "DTE"]].reset_index(drop=True)

c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


**OI session** 2026-10-09 (Final) · **price session** 2026-10-12 · strips 108 · rows 70,335

,STRIP,CONTRACT_CODE,SETTLE_USD_T,OI_LOTS,VOLUME_LOTS,DTE
0,Oct26,V6,278.48,"3,947.00",0,22
1,Nov26,X6,252.98,"3,841.00",80,52
2,Dec26,Z6,235.25,"3,799.00",50,76
3,Jan27,F7,187.50,"2,988.00",35,114
4,Feb27,G7,185.07,"2,998.00",25,142
5,Mar27,H7,182.25,"2,998.00",25,173
6,Apr27,J7,115.76,"1,060.00",20,203
7,May27,K7,115.76,"1,035.00",20,234
8,Jun27,M7,115.76,"1,035.00",20,264
9,Jul27,N7,59.24,630.00,20,295


## Positioning pack (crude-style)

Grey lines = each expired/listed month’s OI path into expiry. Red = current front month. Bars = next 12 listed months; red overlay = day-over-day Δ OI.

In [3]:
tot = total_oi(oi_ok)
front_code = front.iloc[0]["CONTRACT_CODE"]
front_month = front.iloc[0]["CONTRACT_MONTH"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"OI vs days to expiry",
        "Latest OI + Δ (next 12 months)",
        "Total OI",
    ),
    horizontal_spacing=0.07,
)
fig.update_layout(title=f"TD3C (TDL) | {session}", template="plotly_white", height=420, showlegend=False)

# --- spaghetti: last 90 days of each contract's life ---
life = oi_ok.loc[oi_ok["DAYS_TO_EXPIRY"].between(-90, 5)]
for month, g in life.groupby("CONTRACT_MONTH"):
    g = g.sort_values("TRADE_DATE")
    is_front = month == front_month
    fig.add_trace(
        go.Scatter(
            x=g["DAYS_TO_EXPIRY"], y=g["OI_LOTS"],
            mode="lines",
            line=dict(color="#c0392b" if is_front else "#7f8c8d", width=2.4 if is_front else 1),
            opacity=1 if is_front else 0.35,
            hovertemplate="%{x} dte<br>%{y:,.0f} lots",
            name=str(month),
        ),
        row=1, col=1,
    )
fig.update_xaxes(title_text="Days to expiry", row=1, col=1)
fig.update_yaxes(title_text="Open interest", row=1, col=1)

# --- latest OI + delta ---
delta = oi_delta(prev_snap, front)
fig.add_trace(
    go.Bar(x=delta["CONTRACT_CODE"], y=delta["OI_LOTS"], marker_color="#1f4e79",
           hovertemplate="%{x}<br>OI %{y:,.0f}<extra></extra>"),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        x=delta["CONTRACT_CODE"], y=delta["OI_DELTA"].fillna(0),
        marker_color=["#c0392b" if v < 0 else "#27ae60" for v in delta["OI_DELTA"].fillna(0)],
        hovertemplate="Δ %{y:,.0f}<extra></extra>",
    ),
    row=1, col=2,
)
fig.update_xaxes(title_text="Contract", row=1, col=2)
fig.update_yaxes(title_text="Open interest", row=1, col=2)

fig.add_trace(
    go.Scatter(x=tot["TRADE_DATE"], y=tot["OI_LOTS"], mode="lines",
               line=dict(color="#7f8c8d", width=1.6),
               hovertemplate="%{x|%Y-%m-%d}<br>%{y:,.0f} lots"),
    row=1, col=3,
)
fig.update_yaxes(title_text="Total OI", row=1, col=3)
fig.show()

chg = tot.set_index("TRADE_DATE")["OI_LOTS"]
d1 = chg.iloc[-1] - chg.iloc[-2] if len(chg) > 1 else 0
d20 = chg.iloc[-1] - chg.iloc[-21] if len(chg) > 21 else float("nan")
display(Markdown(
    f"Front **{front_code}** · total OI **{chg.iloc[-1]:,.0f}** lots "
    f"({chg.iloc[-1] * 1000 / 270000:,.0f} VLCC-equivalent) · "
    f"Δ1d {d1:+,.0f} · Δ20d {d20:+,.0f} vs {prev_session}."
))

Front **V6** · total OI **35,653** lots (132 VLCC-equivalent) · Δ1d +290 · Δ20d -2,399 vs 2026-10-08.

## Curve and calendar (sentiment on the strip)

FFAs settle on the monthly average of Baltic $/t. A steep backwardation (front >> 12th month) is the market pricing the current Hormuz/war-risk regime to fade — same read as the primer on TC14/TD3C curve shape.

In [4]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Curve $/t | {price_session}", "Front vs 12th month (backwardation)"),
    horizontal_spacing=0.08,
)
fig.update_layout(template="plotly_white", height=400, showlegend=True, legend=dict(orientation="h"))

fig.add_trace(
    go.Scatter(
        x=price_front["CONTRACT_CODE"], y=price_front["SETTLE_USD_T"],
        mode="lines+markers", name="now", line=dict(color="#1f4e79", width=2),
    ),
    row=1, col=1,
)
# curve 20 sessions ago if we have it
hist_dates = sorted(tdl["TRADE_DATE"].unique())
ref = hist_dates[-21] if len(hist_dates) > 21 else hist_dates[0]
ref_front = front_n(tdl.loc[tdl["TRADE_DATE"] == ref], 24, ref)
fig.add_trace(
    go.Scatter(
        x=ref_front["CONTRACT_CODE"], y=ref_front["SETTLE_USD_T"],
        mode="lines", name=str(ref), line=dict(color="#95a5a6", width=1, dash="dot"),
    ),
    row=1, col=1,
)
fig.update_yaxes(title_text="USD/t", row=1, col=1)

rows = []
for d, g in tdl.groupby("TRADE_DATE"):
    live = front_n(g, 12, d)
    if len(live) <= 11:
        continue
    m1, m12 = live.iloc[0]["SETTLE_USD_T"], live.iloc[11]["SETTLE_USD_T"]
    rows.append({"TRADE_DATE": d, "m1": m1, "m12": m12, "spread": m1 - m12})
curve_hist = pd.DataFrame(rows)
fig.add_trace(
    go.Scatter(x=curve_hist["TRADE_DATE"], y=curve_hist["spread"], mode="lines",
               name="M1–M12", line=dict(color="#c0392b")),
    row=1, col=2,
)
fig.add_hline(y=0, line_dash="dot", line_color="#7f8c8d", row=1, col=2)
fig.add_vrect(x0="2026-03-01", x1=str(price_session), fillcolor="#f5b7b1", opacity=0.15,
              line_width=0, row=1, col=2, annotation_text="Hormuz regime")
fig.update_yaxes(title_text="USD/t", row=1, col=2)
fig.show()

m1, m2 = price_front.iloc[0]["SETTLE_USD_T"], price_front.iloc[1]["SETTLE_USD_T"]
display(Markdown(
    f"**{price_front.iloc[0]['CONTRACT_CODE']}** {m1:.2f} vs **{price_front.iloc[1]['CONTRACT_CODE']}** {m2:.2f} "
    f"(M1–M2 {m1-m2:+.2f}) · M1–M12 {price_front.iloc[0]['SETTLE_USD_T'] - price_front.iloc[11]['SETTLE_USD_T']:+.2f} $/t."
))

**V6** 278.66 vs **X6** 260.71 (M1–M2 +17.95) · M1–M12 +219.42 $/t.

## Cash vs paper (daily mini vs remaining month)

`DFN` prompt strip ≈ Baltic TD3C that day in $/t. `TDL` front = FFA for the rest of the month. Gap = paper vs realised-so-far plus remaining-month view (same idea as implied rest-of-month).

In [5]:
spot = dfn.rename(columns={"SETTLE_USD_T": "PROMPT_USD_T"})[["TRADE_DATE", "PROMPT_USD_T"]]
rows = []
for d, g in tdl.groupby("TRADE_DATE"):
    fr = front_n(g, 1, d)
    if fr.empty:
        continue
    rows.append({"TRADE_DATE": d, "M1_USD_T": fr.iloc[0]["SETTLE_USD_T"], "M1": fr.iloc[0]["STRIP"]})
m1_hist = pd.DataFrame(rows)
px = m1_hist.merge(spot, on="TRADE_DATE", how="inner")
px["BASIS"] = px["M1_USD_T"] - px["PROMPT_USD_T"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Prompt DFN vs TDL front", "Paper − prompt (contango of remaining month)"))
fig.update_layout(template="plotly_white", height=520, legend=dict(orientation="h"))
fig.add_trace(go.Scatter(x=px["TRADE_DATE"], y=px["PROMPT_USD_T"], name="DFN prompt", line=dict(color="#7f8c8d")), 1, 1)
fig.add_trace(go.Scatter(x=px["TRADE_DATE"], y=px["M1_USD_T"], name="TDL front", line=dict(color="#1f4e79")), 1, 1)
fig.add_trace(go.Bar(x=px["TRADE_DATE"], y=px["BASIS"], name="basis",
                     marker_color=["#c0392b" if v > 0 else "#27ae60" for v in px["BASIS"]]), 2, 1)
fig.update_yaxes(title_text="USD/t", row=1, col=1)
fig.update_yaxes(title_text="USD/t", row=2, col=1)
fig.show()

last = px.iloc[-1]
display(Markdown(
    f"Last overlap **{last['TRADE_DATE']}**: prompt {last['PROMPT_USD_T']:.2f} vs {last['M1']} {last['M1_USD_T']:.2f} "
    f"(paper {last['BASIS']:+.2f} $/t). Positive = FFA above spot (market sees remaining days softer or already-printed days cheaper)."
))

Last overlap **2026-10-12**: prompt 263.49 vs Oct26 278.66 (paper +15.17 $/t). Positive = FFA above spot (market sees remaining days softer or already-printed days cheaper).

## Liquidity and where the OI sits

Turnover = volume / OI. Concentration in the first three months is how much of the book is still “spot-month trading” vs a structural strip.

In [6]:
conc_rows = []
for d, g in oi_ok.groupby("TRADE_DATE"):
    live = front_n(g, 36, d)
    tot_oi = live["OI_LOTS"].sum()
    if tot_oi <= 0:
        continue
    conc_rows.append({
        "TRADE_DATE": d,
        "FRONT3": live.head(3)["OI_LOTS"].sum() / tot_oi,
        "FRONT12": live.head(12)["OI_LOTS"].sum() / tot_oi,
        "TURNOVER": g["VOLUME_LOTS"].sum() / tot_oi,
        "VOLUME": g["VOLUME_LOTS"].sum(),
        "OI": tot_oi,
    })
conc = pd.DataFrame(conc_rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=("OI share in front 3 / 12 months", "Daily volume vs OI (turnover)"),
                    horizontal_spacing=0.08)
fig.update_layout(template="plotly_white", height=400, legend=dict(orientation="h"))
fig.add_trace(go.Scatter(x=conc["TRADE_DATE"], y=conc["FRONT3"], name="front 3", line=dict(color="#1f4e79")), 1, 1)
fig.add_trace(go.Scatter(x=conc["TRADE_DATE"], y=conc["FRONT12"], name="front 12", line=dict(color="#7f8c8d")), 1, 1)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.add_trace(go.Scatter(x=conc["TRADE_DATE"], y=conc["TURNOVER"], name="vol/OI", line=dict(color="#c0392b")), 1, 2)
fig.show()

c = conc.iloc[-1]
display(Markdown(
    f"On **{c['TRADE_DATE']}** {c['FRONT3']:.0%} of OI is in the first three months, "
    f"{c['FRONT12']:.0%} in the first year. Turnover {c['TURNOVER']:.1%} of outstanding OI that day."
))

On **2026-10-09** 32% of OI is in the first three months, 72% in the first year. Turnover 1.1% of outstanding OI that day.

## Read-through

- **OI falling while the curve stays steeply backwardated** = existing paper being closed/rolled, not a new long-strip bid; consistent with a crisis premium that the back end refuses to own.
- **OI building in the front** into expiry (red spaghetti rising into -20 dte) = short-covering or fresh hedging into the print — watch whether it dumps in the last two weeks (roll) or holds (squeeze).
- **DFN vs TDL**: if prompt prints well above the remaining-month FFA, the monthly average is still catching up (rest-of-month implied lower).
- Next dashboard slice: same pack for **TC14 (WHK)** and **TD20 (TDC)** on one page, plus Platts-settled **WDF** as the USG dirty twin.